<a href="https://colab.research.google.com/github/softmurata/colab_notebooks/blob/main/computervision/Matting_Anything.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# ======================================================
# GPU / CUDA 環境チェック
# GPU ランタイムが必要です:
#   Colab メニュー > ランタイム > ランタイムのタイプを変更
#   > ハードウェア アクセラレータ > GPU (T4) を選択
# ======================================================
import subprocess, os, sys

# nvidia-smi で GPU 確認
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    print("=" * 60)
    print("⚠ 警告: GPU が検出されませんでした")
    print("  Colab メニュー > ランタイム > ランタイムのタイプを変更")
    print("  > ハードウェア アクセラレータ > GPU (T4) を選択してください")
    print("=" * 60)
else:
    name_result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
        capture_output=True, text=True
    )
    print(f"✓ GPU 検出: {name_result.stdout.strip()}")

# CUDA ディレクトリと nvcc の確認
cuda_path = '/usr/local/cuda'
if os.path.exists(cuda_path):
    print(f"✓ CUDA ディレクトリ: {cuda_path}")
    nvcc_result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
    if nvcc_result.returncode == 0:
        version_lines = [l for l in nvcc_result.stdout.split('\n') if 'release' in l.lower()]
        if version_lines:
            print(f"✓ nvcc: {version_lines[0].strip()}")
    else:
        print("⚠ nvcc が見つかりません（CUDA 拡張ビルドに影響する場合があります）")
else:
    print(f"⚠ {cuda_path} が見つかりません")

# torch の CUDA バージョン確認（CUDA_HOME との一致確認に使う）
import torch
print(f"✓ torch.version.cuda : {torch.version.cuda}")
print(f"✓ torch.cuda.is_available(): {torch.cuda.is_available()}")


✓ GPU 検出: Tesla T4, 15360 MiB
✓ CUDA ディレクトリ: /usr/local/cuda
✓ nvcc: Cuda compilation tools, release 12.8, V12.8.93
✓ torch.version.cuda : 12.8
✓ torch.cuda.is_available(): True


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Installation

In [3]:

import os, subprocess, sys, shutil

# -------------------------------------------------------
# CUDA 環境変数を Python から設定
# 重要: !export は別サブシェルで実行されるため次の !コマンドに引き継がれない。
#       os.environ で設定すると !コマンドにも引き継がれる。
# CUDA_HOME のみが GroundingDINO の setup.py の CUDA 判定に使われる。
#   条件: torch.cuda.is_available() and CUDA_HOME is not None
# -------------------------------------------------------
cuda_path = '/usr/local/cuda'
os.environ['CUDA_HOME'] = cuda_path
print(f"✓ 環境変数設定: CUDA_HOME={cuda_path}")

# torch の CUDA バージョンと CUDA_HOME の一致確認
import torch
print(f"  torch.version.cuda : {torch.version.cuda}")
print(f"  torch.cuda.is_available(): {torch.cuda.is_available()}")

# Matting-Anything をクローン（不完全なクローンは削除して再実行）
repo_path = '/content/Matting-Anything'
if not os.path.exists(os.path.join(repo_path, '.git')):
    if os.path.exists(repo_path):
        shutil.rmtree(repo_path)   # 中途半端なディレクトリを削除
    !git clone --depth=1 https://github.com/SHI-Labs/Matting-Anything {repo_path}
%cd /content/Matting-Anything

# 依存関係のインストール
!pip install -r requirements.txt -q

# segment-anything のインストール
!pip install -e segment-anything -q

# GroundingDINO を CUDA 対応でビルド
# --no-build-isolation: 既インストール済みの torch/numpy を使ってビルド（CUDA 拡張に必要）
# -q は外す: CUDA ビルドエラーを隠さないようにする
# 再実行時に確実に再ビルドするため事前にアンインストール
!pip uninstall groundingdino -y -q 2>/dev/null; echo ""
!pip install -e GroundingDINO --no-build-isolation

# GroundingDINO CUDA ops のロード確認（stderr を表示してデバッグしやすくする）
print("\n--- GroundingDINO CUDA ops 確認 ---")
check_result = subprocess.run(
    [sys.executable, '-c', 'import groundingdino._C as _C; print("✓ GroundingDINO CUDA ops: 正常にロードされました (GPU モード)")'],
    capture_output=True, text=True
)
if check_result.returncode == 0:
    print(check_result.stdout.strip())
else:
    print(f"⚠ CUDA ops のロードに失敗しました (CPU モードで実行されます)")
    print(check_result.stderr.strip())

# diffusers のインストール
!pip install --upgrade diffusers[torch] -q

print("\n✓ 全インストール完了")


✓ 環境変数設定: CUDA_HOME=/usr/local/cuda
  torch.version.cuda : 12.8
  torch.cuda.is_available(): True
Cloning into '/content/Matting-Anything'...
remote: Enumerating objects: 131, done.
remote: Counting objects: 100% (131/131), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 131 (delta 11), reused 102 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (131/131), 11.00 MiB | 23.03 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/Matting-Anything
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ...

Download model

In [4]:
# Download GroundingDINO model
!wget https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth -P /content/drive/MyDrive/AI_picasso//Matting-Anything/checkpoints

--2026-05-14 11:49:56--  https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/611591640/f221e500-c2fc-4fd3-b84e-8ad92a6923f3?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-05-14T12%3A25%3A28Z&rscd=attachment%3B+filename%3Dgroundingdino_swint_ogc.pth&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-05-14T11%3A24%3A34Z&ske=2026-05-14T12%3A25%3A28Z&sks=b&skv=2018-11-09&sig=a4PRX8M2GhViHjx9NfX%2FWA3XHKIAjNpIykH79jiHN1c%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3ODc2Mjk5NywibmJmIjoxNzc4NzU5Mzk3LC

In [5]:
import os
# ディレクトリを作成
os.makedirs('/content/Matting-Anything/segment-anything/checkpoints', exist_ok=True)



Start WebUI

In [ ]:
%cd /content/drive/MyDrive/AI_picasso/Matting-Anything
!python gradio_app.py --share

/content/drive/MyDrive/AI_picasso/Matting-Anything
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
final text_encoder_type: bert-base-uncased
config.json: 100% 570/570 [00:00<00:00, 2.23MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 266kB/s]
vocab.txt: 232kB [00:00, 760kB/s]
tokenizer.json: 466kB [00:00, 1.44MB/s]
model.safetensors: 100% 440M/440M [00:03<00:00, 137MB/s]
Loading weights: 100% 199/199 [00:00<00:00, 904.35it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weigh